In [1]:
%pip install -q tensorflow scikit-learn pandas pillow tqdm matplotlib


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print('TF:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

TF: 2.21.0
GPUs: []


In [3]:
BASE_DIR = '.'
IMG_DIR  = os.path.join(BASE_DIR, 'img')

IMG_SIZE     = 224      # bigger = more detail
IMG_CHANNELS = 3

BATCH_SIZE   = 32
EPOCHS       = 80       # scratch CNNs need more time

LR_INIT      = 1e-3

# Regularisation
LABEL_SMOOTH = 0.1
MIXUP_ALPHA  = 0.3
DROPOUT_RATE = 0.4

N_FOLDS  = 5

LABEL_MAP = {0: 'Animal Crossing', 1: 'Doom'}

In [4]:
train_df = pd.read_csv(os.path.join(BASE_DIR, 'train-label.csv'))
test_df  = pd.read_csv(os.path.join(BASE_DIR, 'test-label.csv'))
print('Train:', train_df.shape, '| Test:', test_df.shape)

def load_image_array(filepath, size=IMG_SIZE):
    img = Image.open(filepath).convert('RGB')
    img = img.resize((size, size), Image.BILINEAR)
    return np.array(img, dtype=np.float32)

def load_all(df, img_dir, has_labels=True):
    X, y = [], []
    missing = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Loading'):
        path = os.path.join(img_dir, row['file'])
        if not os.path.exists(path):
            missing += 1; continue
        X.append(load_image_array(path))
        if has_labels:
            y.append(int(row['label']))
    if missing:
        print(f'  ⚠️  {missing} images not found')
    X = np.stack(X).astype(np.float32)
    y = np.array(y, dtype=np.int32) if has_labels else None
    return X, y

print('Loading train ...')
X_all, y_all = load_all(train_df, IMG_DIR, has_labels=True)
print(f'  {X_all.shape}')

print('Loading test ...')
X_test, _ = load_all(test_df, IMG_DIR, has_labels=False)
print(f'  {X_test.shape}')

# Per-channel z-normalization using train stats only
mean = X_all.mean(axis=(0, 1, 2), keepdims=True)
std  = X_all.std(axis=(0, 1, 2), keepdims=True) + 1e-6
X_all  = (X_all  - mean) / std
X_test = (X_test - mean) / std
print(f'Normalized — mean: {mean.flatten()}, std: {std.flatten()}')

Train: (1385, 3) | Test: (1386, 3)
Loading train ...


Loading: 100%|██████████| 1385/1385 [00:26<00:00, 52.08it/s]


  (1385, 224, 224, 3)
Loading test ...


Loading: 100%|██████████| 1386/1386 [00:27<00:00, 50.92it/s]


  (1386, 224, 224, 3)
Normalized — mean: [61.80364 61.80364 61.80364], std: [108.08798  98.08212  93.13737]


In [5]:
def augment_image(image):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.25)
    image = tf.image.random_contrast(image, 0.6, 1.4)
    image = tf.image.random_saturation(image, 0.6, 1.4)
    image = tf.image.random_hue(image, 0.08)

    # Random crop
    pad = int(IMG_SIZE * 0.12)
    image = tf.image.pad_to_bounding_box(
        image, pad, pad, IMG_SIZE + 2*pad, IMG_SIZE + 2*pad)
    image = tf.image.random_crop(image, [IMG_SIZE, IMG_SIZE, IMG_CHANNELS])

    # Random rotation ±30°
    angle = tf.random.uniform([], -0.52, 0.52)  # radians
    image = tf.keras.preprocessing.image.apply_affine_transform(
        image.numpy(), theta=angle * 180 / 3.14159)
    image = tf.cast(image, tf.float32)

    return image

def mixup_batch(images, labels_onehot, alpha=MIXUP_ALPHA):
    batch_size = tf.shape(images)[0]
    lam = tf.random.uniform([batch_size], minval=1.0 - alpha, maxval=1.0)
    lam = tf.reshape(lam, [-1, 1, 1, 1])
    idx = tf.random.shuffle(tf.range(batch_size))
    images2 = tf.gather(images, idx)
    labels2 = tf.gather(labels_onehot, idx)
    mixed_images = lam * images + (1 - lam) * images2
    lam_label = tf.reshape(lam, [-1, 1])
    mixed_labels = lam_label * labels_onehot + (1 - lam_label) * labels2
    return mixed_images, mixed_labels

def make_train_ds(X, y, batch_size=BATCH_SIZE, seed=SEED, use_mixup=True):
    AUTOTUNE = tf.data.AUTOTUNE
    num_classes = 2

    ds = (tf.data.Dataset
          .from_tensor_slices((X, y))
          .shuffle(len(X), seed=seed)
          .batch(batch_size))

    if use_mixup:
        def apply_mixup(images, labels):
            labels_oh = tf.one_hot(labels, num_classes)
            return mixup_batch(images, labels_oh)
        ds = ds.map(apply_mixup, num_parallel_calls=AUTOTUNE)

    return ds.prefetch(AUTOTUNE)

def make_eval_ds(X, y=None, batch_size=BATCH_SIZE):
    AUTOTUNE = tf.data.AUTOTUNE
    if y is not None:
        ds = tf.data.Dataset.from_tensor_slices((X, y))
    else:
        ds = tf.data.Dataset.from_tensor_slices(X)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

print('Augmentation pipeline ready.')

Augmentation pipeline ready.


In [6]:
class BalancedAccuracyCallback(keras.callbacks.Callback):
    def __init__(self, val_ds, val_labels):
        super().__init__()
        self.val_ds = val_ds
        self.val_labels = val_labels
        self.best_bal_acc = 0.0
        self.best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        probs = self.model.predict(self.val_ds, verbose=0)
        preds = np.argmax(probs, axis=1)
        bal_acc = balanced_accuracy_score(self.val_labels, preds)
        logs['val_bal_acc'] = bal_acc
        if bal_acc > self.best_bal_acc:
            self.best_bal_acc = bal_acc
            self.best_weights = self.model.get_weights()
        if (epoch + 1) % 10 == 0:
            print(f'    Epoch {epoch+1}: val_bal_acc={bal_acc:.4f} (best={self.best_bal_acc:.4f})')

In [7]:
def se_block(x, ratio=8):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(max(filters // ratio, 1), activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    se = layers.Reshape((1, 1, filters))(se)
    return layers.Multiply()([x, se])

def residual_block(x, filters, stride=1, dropout=0.0):
    shortcut = x
    y = layers.Conv2D(filters, 3, strides=stride, padding='same',
                      use_bias=False, kernel_initializer='he_normal')(x)
    y = layers.BatchNormalization()(y)
    y = layers.Activation('relu')(y)
    if dropout > 0:
        y = layers.Dropout(dropout)(y)
    y = layers.Conv2D(filters, 3, padding='same',
                      use_bias=False, kernel_initializer='he_normal')(y)
    y = layers.BatchNormalization()(y)
    y = se_block(y)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same',
                                 use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    y = layers.Add()([y, shortcut])
    y = layers.Activation('relu')(y)
    return y

def build_model():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, IMG_CHANNELS))

    # Stem
    x = layers.Conv2D(32, 3, strides=2, padding='same',
                      use_bias=False, kernel_initializer='he_normal')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Stage 1
    x = residual_block(x, 64,  stride=2, dropout=0.1)
    x = residual_block(x, 64,  dropout=0.1)

    # Stage 2
    x = residual_block(x, 128, stride=2, dropout=0.2)
    x = residual_block(x, 128, dropout=0.2)

    # Stage 3
    x = residual_block(x, 256, stride=2, dropout=0.3)
    x = residual_block(x, 256, dropout=0.3)

    # Stage 4
    x = residual_block(x, 512, stride=2, dropout=0.3)
    x = residual_block(x, 512, dropout=0.3)

    # Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    x = layers.Dense(256, activation='relu', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(2, activation='softmax')(x)

    return keras.Model(inputs, outputs)

def smoothed_loss(y_true, y_pred):
    if len(y_true.shape) == 1:
        y_true = tf.one_hot(tf.cast(y_true, tf.int32), 2)
    return keras.losses.categorical_crossentropy(
        y_true, y_pred, label_smoothing=LABEL_SMOOTH)

print('Model builder ready.')
m = build_model()
print(f'Total params: {m.count_params():,}')
del m

Model builder ready.
Total params: 11,472,466


In [8]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_probs  = np.zeros((len(X_all), 2), dtype=np.float32)
test_probs = np.zeros((len(X_test), 2), dtype=np.float32)
fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_all, y_all)):
    print(f'\n{"="*55}')
    print(f'  FOLD {fold+1}/{N_FOLDS}')
    print(f'{"="*55}')

    X_tr, X_val = X_all[tr_idx], X_all[val_idx]
    y_tr, y_val = y_all[tr_idx], y_all[val_idx]

    train_ds = make_train_ds(X_tr, y_tr, use_mixup=True)
    val_ds   = make_eval_ds(X_val, y_val)
    test_ds  = make_eval_ds(X_test)

    model = build_model()

    total_steps = (len(X_tr) // BATCH_SIZE) * EPOCHS
    lr_schedule = keras.optimizers.schedules.CosineDecayRestarts(
        initial_learning_rate=LR_INIT,
        first_decay_steps=total_steps // 3,
        t_mul=1.0,
        alpha=1e-6
    )

    model.compile(
        optimizer=keras.optimizers.Adam(lr_schedule),
        loss=smoothed_loss,
        metrics=['accuracy']
    )

    bal_cb = BalancedAccuracyCallback(val_ds, y_val)
    model.fit(train_ds, epochs=EPOCHS, validation_data=val_ds,
              callbacks=[bal_cb], verbose=0)
    model.set_weights(bal_cb.best_weights)
    print(f'  Best val_bal_acc: {bal_cb.best_bal_acc:.4f}')

    # OOF
    oof_probs[val_idx] = model.predict(make_eval_ds(X_val), verbose=0)
    fold_ba = balanced_accuracy_score(y_val, np.argmax(oof_probs[val_idx], axis=1))
    fold_scores.append(fold_ba)
    print(f'  → Fold {fold+1} OOF Balanced Accuracy: {fold_ba:.4f}')

    # TTA: clean + hflip
    raw   = model.predict(test_ds, verbose=0)
    hflip = model.predict(make_eval_ds(X_test[:, :, ::-1, :]), verbose=0)
    test_probs += (raw + hflip) / 2 / N_FOLDS

    del model
    keras.backend.clear_session()

print(f'\n{"="*55}')
for i, s in enumerate(fold_scores):
    print(f'  Fold {i+1}: {s:.4f}')
print(f'  Mean OOF BA: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')


  FOLD 1/5
    Epoch 10: val_bal_acc=0.7382 (best=0.7510)
    Epoch 20: val_bal_acc=0.7785 (best=0.7785)
    Epoch 30: val_bal_acc=0.6079 (best=0.7915)
    Epoch 40: val_bal_acc=0.7250 (best=0.7915)
    Epoch 50: val_bal_acc=0.7338 (best=0.7915)
    Epoch 60: val_bal_acc=0.6809 (best=0.7915)
    Epoch 70: val_bal_acc=0.7416 (best=0.7915)
    Epoch 80: val_bal_acc=0.6804 (best=0.7915)
  Best val_bal_acc: 0.7915
  → Fold 1 OOF Balanced Accuracy: 0.7915

  FOLD 2/5
    Epoch 10: val_bal_acc=0.6870 (best=0.7414)
    Epoch 20: val_bal_acc=0.7731 (best=0.7731)
    Epoch 30: val_bal_acc=0.7189 (best=0.7731)
    Epoch 40: val_bal_acc=0.7473 (best=0.7731)


KeyboardInterrupt: 

In [ ]:
oof_preds = np.argmax(oof_probs, axis=1)
overall_ba = balanced_accuracy_score(y_all, oof_preds)
print(f'Overall OOF Balanced Accuracy: {overall_ba:.4f}')
for cls, name in [(0, 'Animal Crossing'), (1, 'Doom')]:
    mask = y_all == cls
    print(f'  {name}: {(oof_preds[mask]==y_all[mask]).mean():.4f}')

confidence = np.max(test_probs, axis=1)
print(f'\nMean test confidence: {confidence.mean():.3f}')
print(f'Low-confidence (<0.6): {(confidence < 0.6).sum()} / {len(confidence)}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar([f'Fold {i+1}' for i in range(N_FOLDS)], fold_scores,
            color='#0652DD', edgecolor='black', alpha=0.8)
axes[0].axhline(np.mean(fold_scores), color='red', linestyle='--',
                label=f'Mean={np.mean(fold_scores):.4f}')
axes[0].set_ylim(0, 1.05); axes[0].legend()
axes[0].set_title('OOF Balanced Accuracy per Fold')

axes[1].hist(confidence, bins=40, color='#0652DD', alpha=0.8)
axes[1].axvline(0.6, color='red', linestyle='--', label='0.6')
axes[1].set_title('Test Confidence Distribution'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
test_preds = np.argmax(test_probs, axis=1)
submission = pd.DataFrame({'id': test_df['id'].values, 'label': test_preds})
submission.to_csv('sub15.csv', index=False)

print(submission['label'].value_counts())
print(f'Saved sub15.csv — {len(submission)} rows')
assert len(submission) == len(test_df)
assert set(submission['label'].unique()).issubset({0,1})
print('✅ Format OK')